# Taxi Trip Fetching

In this notebook we fetch the taxi trip data provided by the city of chicago: https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data (downloaded on 05.05.2026).

In [1]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd

import requests

# weather data
import openmeteo_requests
import requests_cache
from retry_requests import retry

import time

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Fetch weather data					  #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 41.8781,
	"longitude": 87.6298,
	"start_date": "2024-01-01",
	"end_date": "2026-01-01",
	"daily": ["sunrise", "sunset", "daylight_duration", "sunshine_duration"],
	"hourly": ["temperature_2m", "relative_humidity_2m", "apparent_temperature", "precipitation", "rain", "snowfall", "snow_depth", "surface_pressure", "cloud_cover", "wind_speed_10m", "wind_speed_100m", "is_day", "sunshine_duration", "direct_radiation"],
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(2).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(3).ValuesAsNumpy()
hourly_rain = hourly.Variables(4).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(5).ValuesAsNumpy()
hourly_snow_depth = hourly.Variables(6).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(7).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(8).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(9).ValuesAsNumpy()
hourly_wind_speed_100m = hourly.Variables(10).ValuesAsNumpy()
hourly_is_day = hourly.Variables(11).ValuesAsNumpy()
hourly_sunshine_duration = hourly.Variables(12).ValuesAsNumpy()
hourly_direct_radiation = hourly.Variables(13).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["snowfall"] = hourly_snowfall
hourly_data["snow_depth"] = hourly_snow_depth
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_speed_100m"] = hourly_wind_speed_100m
hourly_data["is_day"] = hourly_is_day
hourly_data["sunshine_duration"] = hourly_sunshine_duration
hourly_data["direct_radiation"] = hourly_direct_radiation

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_sunrise = daily.Variables(0).ValuesInt64AsNumpy()
daily_sunset = daily.Variables(1).ValuesInt64AsNumpy()
daily_daylight_duration = daily.Variables(2).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(3).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["sunrise"] = daily_sunrise
daily_data["sunset"] = daily_sunset
daily_data["daylight_duration"] = daily_daylight_duration
daily_data["sunshine_duration"] = daily_sunshine_duration

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)


OpenMeteoRequestsError: failed to request 'https://archive-api.open-meteo.com/v1/archive': HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Max retries exceeded with url: /v1/archive?latitude=41.8781&longitude=87.6298&start_date=2024-01-01&end_date=2026-01-01&daily=sunrise&daily=sunset&daily=daylight_duration&daily=sunshine_duration&hourly=temperature_2m&hourly=relative_humidity_2m&hourly=apparent_temperature&hourly=precipitation&hourly=rain&hourly=snowfall&hourly=snow_depth&hourly=surface_pressure&hourly=cloud_cover&hourly=wind_speed_10m&hourly=wind_speed_100m&hourly=is_day&hourly=sunshine_duration&hourly=direct_radiation&timezone=auto&format=flatbuffers (Caused by ResponseError('too many 504 error responses'))

In [ ]:
daily_dataframe.head()

,date,sunrise,sunset,daylight_duration,sunshine_duration
0,2024-01-01 00:00:00+00:00,1704073012,1704106094,33078.183594,30482.771484
1,2024-01-02 00:00:00+00:00,1704159418,1704192545,33123.246094,30082.148438
2,2024-01-03 00:00:00+00:00,1704245822,1704278999,33172.207031,28800.000000
3,2024-01-04 00:00:00+00:00,1704332224,1704365454,33224.976562,30764.603516
4,2024-01-05 00:00:00+00:00,1704418624,1704451910,33281.449219,31103.152344


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Permanently Store Weather Data          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# convert timestamps to datetime to avoid false interpretation of dates
daily_dataframe["date"] = pd.to_datetime(daily_dataframe["date"], utc=True)
hourly_dataframe["date"] = pd.to_datetime(hourly_dataframe["date"], utc=True)

# convert sunrise and sunset times from iso8601 to datetime
daily_dataframe["sunrise"] = pd.to_datetime(daily_dataframe["sunrise"], unit="s", utc=True)
daily_dataframe["sunset"] = pd.to_datetime(daily_dataframe["sunset"], unit="s", utc=True)

#daily_dataframe.to_csv("../../data/weather/weather_daily_test.csv", index=False)
#hourly_dataframe.to_csv("../../data/weather/weather_hourly_test.csv", index=False)

In [ ]:
hourly_dataframe_march = hourly_dataframe[( hourly_dataframe["date"] >= 2025-03-01) & (hourly_dataframe["date"] < 2025-04-01)]